# Advanced Problems with Solutions: Custom JSON Serialization in Python

This notebook extends custom JSON serialization into production-style design problems.

## What you will practice

- Designing explicit JSON representations for non-native Python types
- `json.dumps(..., default=...)`
- `functools.singledispatch`
- `json.JSONEncoder`
- `object_hook` and safe decoding
- Timezone-aware `datetime` handling
- Exact `Decimal` and `Fraction` preservation
- `complex`, `bytes`, `UUID`, `Enum`, `Path`, sets, and dataclasses
- Deterministic/canonical JSON
- Versioned payloads and migrations
- Atomic file writes
- JSON Lines streaming
- Security boundaries for deserialization
- Property-style round-trip testing
- A capstone codec for nested domain objects

> **Best-practice rule:** Serialization is a data-contract problem, not just a formatting problem. Prefer explicit schemas, explicit type tags, strict failures, timezone-aware timestamps, and whitelist-based decoding.

## Best-practices checklist

1. **Fail closed.** Unknown objects should normally raise `TypeError`; silently calling `str(obj)` can hide data loss.
2. **Use timezone-aware datetimes.** Prefer UTC and serialize with an explicit offset or `Z`.
3. **Preserve numeric intent.** Avoid converting `Decimal` or `Fraction` to binary floats if exactness matters.
4. **Use explicit type tags for round trips.** Example: `{"__type__": "decimal", "value": "0.10"}`.
5. **Never use `eval()` to reconstruct objects from JSON.**
6. **Whitelist decodable types.** Do not import or instantiate arbitrary classes named in untrusted JSON.
7. **Separate encoding from domain logic when possible.**
8. **Make deterministic output explicit** when JSON is hashed, signed, cached, diffed, or tested.
9. **Version long-lived payloads.** A serializer is an API; formats evolve.
10. **Test invariants, not only examples.** Check types, equality, precision, ordering assumptions, and failure behavior.

## Setup

In [1]:
import base64
import json
import math
import os
import tempfile
import timeit
import uuid

from dataclasses import dataclass, fields, is_dataclass
from datetime import date, datetime, time, timezone, timedelta
from decimal import Decimal
from enum import Enum
from fractions import Fraction
from functools import singledispatch
from pathlib import Path
from typing import Any

In [2]:
class Status(Enum):
    NEW = "new"
    ACTIVE = "active"
    CLOSED = "closed"


@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str


@dataclass
class User:
    user_id: uuid.UUID
    name: str
    created_at: datetime
    status: Status
    tags: set[str]

# Problem 1 — Build a strict multi-type serializer

You receive a nested payload containing objects that Python's default JSON encoder cannot serialize:

- `datetime`
- `date`
- `time`
- `Decimal`
- `Fraction`
- `complex`
- `bytes`
- `UUID`
- `Path`
- `set` / `frozenset`
- `Enum`

Design a `default=` callable using `@singledispatch`.

### Requirements

- Preserve enough information for a later round trip.
- Use explicit `__type__` tags.
- Do **not** silently call `str(obj)` for unsupported types.
- Reject naive `datetime` objects.
- Encode binary data with Base64.

In [3]:
# Starter skeleton — intentionally incomplete.

@singledispatch
def encode_json_value(obj):
    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")

# TODO:
# @encode_json_value.register(...)
# def _(obj):
#     ...

### Solution

In [4]:
TYPE_KEY = "__type__"


def _utc_iso(dt: datetime) -> str:
    """Return an RFC-3339-style UTC timestamp ending in Z."""
    if dt.tzinfo is None or dt.utcoffset() is None:
        raise ValueError("Naive datetime is not allowed; attach a timezone first.")
    dt_utc = dt.astimezone(timezone.utc)
    return dt_utc.isoformat().replace("+00:00", "Z")


@singledispatch
def encode_json_value(obj):
    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")


@encode_json_value.register(datetime)
def _(obj):
    return {TYPE_KEY: "datetime", "value": _utc_iso(obj)}


@encode_json_value.register(date)
def _(obj):
    return {TYPE_KEY: "date", "value": obj.isoformat()}


@encode_json_value.register(time)
def _(obj):
    return {TYPE_KEY: "time", "value": obj.isoformat()}


@encode_json_value.register(Decimal)
def _(obj):
    return {TYPE_KEY: "decimal", "value": str(obj)}


@encode_json_value.register(Fraction)
def _(obj):
    return {
        TYPE_KEY: "fraction",
        "numerator": obj.numerator,
        "denominator": obj.denominator,
    }


@encode_json_value.register(complex)
def _(obj):
    return {TYPE_KEY: "complex", "real": obj.real, "imag": obj.imag}


@encode_json_value.register(bytes)
def _(obj):
    encoded = base64.b64encode(obj).decode("ascii")
    return {TYPE_KEY: "bytes", "encoding": "base64", "value": encoded}


@encode_json_value.register(uuid.UUID)
def _(obj):
    return {TYPE_KEY: "uuid", "value": str(obj)}


@encode_json_value.register(Path)
def _(obj):
    return {TYPE_KEY: "path", "value": str(obj)}


@encode_json_value.register(set)
def _(obj):
    return {TYPE_KEY: "set", "items": list(obj)}


@encode_json_value.register(frozenset)
def _(obj):
    return {TYPE_KEY: "frozenset", "items": list(obj)}


@encode_json_value.register(Enum)
def _(obj):
    return {
        TYPE_KEY: "enum",
        "enum": type(obj).__name__,
        "value": obj.value,
    }

In [5]:
sample_payload = {
    "when": datetime(2026, 8, 7, 12, 30, tzinfo=timezone.utc),
    "birthday": date(1999, 12, 31),
    "alarm": time(8, 15, 30),
    "price": Decimal("19.9900"),
    "ratio": Fraction(2, 7),
    "z": 3 + 4j,
    "blob": b"\x00\x01JSON\xff",
    "request_id": uuid.UUID("12345678-1234-5678-1234-567812345678"),
    "path": Path("/tmp/example.json"),
    "roles": {"admin", "editor"},
    "frozen": frozenset({1, 2, 3}),
    "status": Status.ACTIVE,
}

encoded = json.dumps(sample_payload, default=encode_json_value, indent=2, sort_keys=True)
print(encoded)

{
  "alarm": {
    "__type__": "time",
    "value": "08:15:30"
  },
  "birthday": {
    "__type__": "date",
    "value": "1999-12-31"
  },
  "blob": {
    "__type__": "bytes",
    "encoding": "base64",
    "value": "AAFKU09O/w=="
  },
  "frozen": {
    "__type__": "frozenset",
    "items": [
      1,
      2,
      3
    ]
  },
  "path": {
    "__type__": "path",
    "value": "\\tmp\\example.json"
  },
  "price": {
    "__type__": "decimal",
    "value": "19.9900"
  },
  "ratio": {
    "__type__": "fraction",
    "denominator": 7,
    "numerator": 2
  },
  "request_id": {
    "__type__": "uuid",
    "value": "12345678-1234-5678-1234-567812345678"
  },
  "roles": {
    "__type__": "set",
    "items": [
      "editor",
      "admin"
    ]
  },
  "status": {
    "__type__": "enum",
    "enum": "Status",
    "value": "active"
  },
  "when": {
    "__type__": "datetime",
    "value": "2026-08-07T12:30:00Z"
  },
  "z": {
    "__type__": "complex",
    "imag": 4.0,
    "real": 3.0
  }
}


### Why this is better than `return str(obj)`

A blanket string fallback makes serialization *appear* successful while losing type information. For example, `"0.10"` could have originated from a `Decimal`, a string, or a custom class. A strict serializer surfaces unsupported types early and makes the JSON contract inspectable.

# Problem 2 — Write a safe round-trip decoder

Implement an `object_hook` that reconstructs the tagged values from Problem 1.

### Requirements

- Decode only known tags.
- Whitelist Enum classes.
- Reject malformed Base64.
- Convert `Z` timestamps back to aware UTC `datetime`.
- Leave ordinary dictionaries untouched.

In [6]:
# Starter skeleton.

ENUM_REGISTRY = {
    "Status": Status,
}

def decode_json_object(obj):
    # TODO
    return obj

### Solution

In [7]:
ENUM_REGISTRY = {
    "Status": Status,
}


def _parse_datetime(value: str) -> datetime:
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"
    result = datetime.fromisoformat(value)
    if result.tzinfo is None or result.utcoffset() is None:
        raise ValueError("Decoded datetime must be timezone-aware.")
    return result.astimezone(timezone.utc)


def decode_json_object(obj):
    tag = obj.get(TYPE_KEY)
    if tag is None:
        return obj

    if tag == "datetime":
        return _parse_datetime(obj["value"])

    if tag == "date":
        return date.fromisoformat(obj["value"])

    if tag == "time":
        return time.fromisoformat(obj["value"])

    if tag == "decimal":
        return Decimal(obj["value"])

    if tag == "fraction":
        return Fraction(obj["numerator"], obj["denominator"])

    if tag == "complex":
        return complex(obj["real"], obj["imag"])

    if tag == "bytes":
        if obj.get("encoding") != "base64":
            raise ValueError("Unsupported bytes encoding.")
        return base64.b64decode(obj["value"], validate=True)

    if tag == "uuid":
        return uuid.UUID(obj["value"])

    if tag == "path":
        return Path(obj["value"])

    if tag == "set":
        return set(obj["items"])

    if tag == "frozenset":
        return frozenset(obj["items"])

    if tag == "enum":
        enum_name = obj["enum"]
        enum_cls = ENUM_REGISTRY.get(enum_name)
        if enum_cls is None:
            raise ValueError(f"Enum is not whitelisted: {enum_name!r}")
        return enum_cls(obj["value"])

    raise ValueError(f"Unknown tagged JSON type: {tag!r}")

In [8]:
decoded = json.loads(encoded, object_hook=decode_json_object)

assert decoded["when"] == sample_payload["when"]
assert isinstance(decoded["when"], datetime)
assert decoded["price"] == Decimal("19.9900")
assert isinstance(decoded["price"], Decimal)
assert decoded["ratio"] == Fraction(2, 7)
assert decoded["z"] == 3 + 4j
assert decoded["blob"] == b"\x00\x01JSON\xff"
assert decoded["request_id"] == sample_payload["request_id"]
assert decoded["path"] == Path("/tmp/example.json")
assert decoded["roles"] == {"admin", "editor"}
assert decoded["frozen"] == frozenset({1, 2, 3})
assert decoded["status"] is Status.ACTIVE

print("Problem 2 round-trip checks passed.")

Problem 2 round-trip checks passed.


# Problem 3 — Make sets deterministic

JSON arrays are ordered, but Python sets are not. If the same logical payload is serialized twice, arbitrary set iteration can produce different JSON text.

This matters for tests, cache keys, hashing, signatures, content-addressed storage, and reproducible builds.

Design a deterministic representation for sets even when their members are heterogeneous and cannot be directly compared.

### Solution

In [9]:
def stable_json_sort_key(value: Any) -> str:
    return json.dumps(
        value,
        default=encode_json_value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )


@encode_json_value.register(set)
def _(obj):
    items = sorted(obj, key=stable_json_sort_key)
    return {TYPE_KEY: "set", "items": items}


@encode_json_value.register(frozenset)
def _(obj):
    items = sorted(obj, key=stable_json_sort_key)
    return {TYPE_KEY: "frozenset", "items": items}

In [10]:
heterogeneous = {
    "values": {
        10,
        "10",
        Decimal("10.0"),
        Fraction(1, 3),
        uuid.UUID("aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa"),
    }
}

json_1 = json.dumps(
    heterogeneous,
    default=encode_json_value,
    sort_keys=True,
    separators=(",", ":"),
)

json_2 = json.dumps(
    heterogeneous,
    default=encode_json_value,
    sort_keys=True,
    separators=(",", ":"),
)

assert json_1 == json_2
print(json_1)

{"values":{"__type__":"set","items":["10",10,{"__type__":"fraction","denominator":3,"numerator":1},{"__type__":"uuid","value":"aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa"}]}}


# Problem 4 — Handle dataclasses without serializing private implementation details

A tempting fallback is `vars(obj)`, but that can accidentally serialize caches, passwords/tokens, lazy internal state, file handles, or framework-specific fields.

Create explicit dataclass support with a whitelist and public field selection.

We will serialize `Money` and `User`.

### Solution

In [11]:
DATACLASS_REGISTRY = {}


def register_dataclass_type(cls):
    if not is_dataclass(cls):
        raise TypeError(f"{cls!r} is not a dataclass type")
    DATACLASS_REGISTRY[cls.__name__] = cls
    return cls


register_dataclass_type(Money)
register_dataclass_type(User)


@encode_json_value.register(object)
def _(obj):
    if is_dataclass(obj) and not isinstance(obj, type):
        cls = type(obj)
        if cls.__name__ not in DATACLASS_REGISTRY:
            raise TypeError(f"Dataclass is not whitelisted: {cls.__name__}")

        public_fields = {
            field.name: getattr(obj, field.name)
            for field in fields(obj)
            if not field.name.startswith("_")
        }

        return {
            TYPE_KEY: "dataclass",
            "class": cls.__name__,
            "fields": public_fields,
        }

    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")

In [12]:
_decode_json_object_without_dataclasses = decode_json_object

def decode_json_object(obj):
    if obj.get(TYPE_KEY) == "dataclass":
        class_name = obj["class"]
        cls = DATACLASS_REGISTRY.get(class_name)
        if cls is None:
            raise ValueError(f"Dataclass is not whitelisted: {class_name!r}")
        return cls(**obj["fields"])

    return _decode_json_object_without_dataclasses(obj)

In [13]:
user = User(
    user_id=uuid.UUID("bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb"),
    name="Ada",
    created_at=datetime(2026, 8, 7, 10, 0, tzinfo=timezone.utc),
    status=Status.NEW,
    tags={"python", "json"},
)

money = Money(Decimal("1234.50"), "EUR")

payload = {"user": user, "balance": money}
text = json.dumps(payload, default=encode_json_value, indent=2, sort_keys=True)
print(text)

restored = json.loads(text, object_hook=decode_json_object)
assert restored["user"] == user
assert restored["balance"] == money
print("Dataclass round trip passed.")

{
  "balance": {
    "__type__": "dataclass",
    "class": "Money",
    "fields": {
      "amount": {
        "__type__": "decimal",
        "value": "1234.50"
      },
      "currency": "EUR"
    }
  },
  "user": {
    "__type__": "dataclass",
    "class": "User",
    "fields": {
      "created_at": {
        "__type__": "datetime",
        "value": "2026-08-07T10:00:00Z"
      },
      "name": "Ada",
      "status": {
        "__type__": "enum",
        "enum": "Status",
        "value": "new"
      },
      "tags": {
        "__type__": "set",
        "items": [
          "json",
          "python"
        ]
      },
      "user_id": {
        "__type__": "uuid",
        "value": "bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb"
      }
    }
  }
}
Dataclass round trip passed.


# Problem 5 — Reject naive datetimes and normalize offsets

A timestamp without timezone information is ambiguous. Two systems can interpret the same wall-clock time differently.

Write tests showing that:

1. naive datetimes are rejected,
2. non-UTC aware datetimes are normalized to UTC,
3. decoded timestamps remain timezone-aware.

### Solution

In [14]:
naive = datetime(2026, 8, 7, 12, 0)

try:
    json.dumps({"dt": naive}, default=encode_json_value)
except ValueError as exc:
    print("Expected failure:", exc)
else:
    raise AssertionError("Naive datetime should have failed.")

Expected failure: Naive datetime is not allowed; attach a timezone first.


In [15]:
plus_three = timezone(timedelta(hours=3))
local_dt = datetime(2026, 8, 7, 15, 30, tzinfo=plus_three)

text = json.dumps({"dt": local_dt}, default=encode_json_value)
print(text)

restored = json.loads(text, object_hook=decode_json_object)
assert restored["dt"] == datetime(2026, 8, 7, 12, 30, tzinfo=timezone.utc)
assert restored["dt"].tzinfo is not None
print("Timezone normalization passed.")

{"dt": {"__type__": "datetime", "value": "2026-08-07T12:30:00Z"}}
Timezone normalization passed.


# Problem 6 — Preserve `Decimal` precision

Compare three approaches:

1. Convert `Decimal` to `float`
2. Serialize it as a plain JSON string
3. Serialize it with an explicit type tag

Explain which approaches preserve both **value** and **type**.

### Solution

In [16]:
d = Decimal("0.123456789012345678901234567890")

as_float = float(d)
print("Decimal:", d)
print("float:  ", repr(as_float))

assert Decimal(str(as_float)) != d

Decimal: 0.123456789012345678901234567890
float:   0.12345678901234568


In [17]:
plain_string_json = json.dumps({"value": str(d)})
plain_string_decoded = json.loads(plain_string_json)

assert plain_string_decoded["value"] == str(d)
assert isinstance(plain_string_decoded["value"], str)

tagged_json = json.dumps({"value": d}, default=encode_json_value)
tagged_decoded = json.loads(tagged_json, object_hook=decode_json_object)

assert tagged_decoded["value"] == d
assert isinstance(tagged_decoded["value"], Decimal)

print("Plain string preserves digits but not type.")
print("Tagged representation preserves digits and Decimal type.")

Plain string preserves digits but not type.
Tagged representation preserves digits and Decimal type.


### Bonus: `parse_float=Decimal`

For ordinary JSON numbers, `json.loads(..., parse_float=Decimal)` can preserve decimal text more faithfully than binary float parsing.

In [18]:
numeric_json = '{"price": 0.10, "tax": 0.075}'
parsed_default = json.loads(numeric_json)
parsed_decimal = json.loads(numeric_json, parse_float=Decimal)

print(parsed_default, {k: type(v).__name__ for k, v in parsed_default.items()})
print(parsed_decimal, {k: type(v).__name__ for k, v in parsed_decimal.items()})

assert parsed_decimal["price"] == Decimal("0.10")

{'price': 0.1, 'tax': 0.075} {'price': 'float', 'tax': 'float'}
{'price': Decimal('0.10'), 'tax': Decimal('0.075')} {'price': 'Decimal', 'tax': 'Decimal'}


# Problem 7 — Implement the same codec with `json.JSONEncoder`

Some codebases prefer an encoder class so serialization policy can be passed around as `cls=...`.

Create `AdvancedJSONEncoder` that delegates to the same `encode_json_value()` function.

### Solution

In [19]:
class AdvancedJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        try:
            return encode_json_value(obj)
        except TypeError:
            return super().default(obj)

In [20]:
payload = {
    "time": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
    "money": Money(Decimal("9.99"), "USD"),
    "flags": {"a", "b"},
}

text = json.dumps(
    payload,
    cls=AdvancedJSONEncoder,
    indent=2,
    sort_keys=True,
)
print(text)

round_tripped = json.loads(text, object_hook=decode_json_object)
assert round_tripped["money"] == payload["money"]
assert round_tripped["flags"] == payload["flags"]

{
  "flags": {
    "__type__": "set",
    "items": [
      "a",
      "b"
    ]
  },
  "money": {
    "__type__": "dataclass",
    "class": "Money",
    "fields": {
      "amount": {
        "__type__": "decimal",
        "value": "9.99"
      },
      "currency": "USD"
    }
  },
  "time": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  }
}


# Problem 8 — Canonical JSON for hashing/signing

Build a helper `canonical_json(obj)` with these properties:

- UTF-8 friendly: `ensure_ascii=False`
- No insignificant whitespace
- Sorted object keys
- Deterministic set ordering via our encoder
- Reject non-standard NaN and Infinity

Then demonstrate why `allow_nan=False` is important.

### Solution

In [21]:
def canonical_json(obj: Any) -> str:
    return json.dumps(
        obj,
        default=encode_json_value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )


canonical_payload = {
    "message": "Здравей, JSON 👋",
    "values": {"b", "a", "c"},
    "amount": Decimal("10.00"),
}

canonical_text = canonical_json(canonical_payload)
print(canonical_text)

{"amount":{"__type__":"decimal","value":"10.00"},"message":"Здравей, JSON 👋","values":{"__type__":"set","items":["a","b","c"]}}


In [22]:
for bad_number in [math.nan, math.inf, -math.inf]:
    try:
        canonical_json({"value": bad_number})
    except ValueError as exc:
        print("Rejected:", bad_number, "->", exc)
    else:
        raise AssertionError("Non-standard JSON number should have been rejected.")

Rejected: nan -> Out of range float values are not JSON compliant: nan
Rejected: inf -> Out of range float values are not JSON compliant: inf
Rejected: -inf -> Out of range float values are not JSON compliant: -inf


# Problem 9 — Add schema versioning and migration

Long-lived JSON formats evolve. Suppose version 1 stored a user name as one `"name"` field, while version 2 stores `"first_name"` and `"last_name"`.

Design:

- a versioned envelope,
- a migration from v1 to v2,
- strict rejection of unknown future versions.

### Solution

In [23]:
CURRENT_SCHEMA_VERSION = 2


def make_envelope(payload: dict, version: int = CURRENT_SCHEMA_VERSION) -> dict:
    return {
        "schema": "user-event",
        "version": version,
        "payload": payload,
    }


def migrate_envelope(envelope: dict) -> dict:
    if envelope.get("schema") != "user-event":
        raise ValueError("Unexpected schema name.")

    version = envelope.get("version")

    if version == 1:
        migrated = dict(envelope)
        payload = dict(migrated["payload"])

        full_name = payload.pop("name")
        first_name, _, last_name = full_name.partition(" ")

        payload["first_name"] = first_name
        payload["last_name"] = last_name

        migrated["payload"] = payload
        migrated["version"] = 2
        envelope = migrated
        version = 2

    if version != CURRENT_SCHEMA_VERSION:
        raise ValueError(
            f"Unsupported schema version {version!r}; "
            f"expected {CURRENT_SCHEMA_VERSION}."
        )

    return envelope

In [24]:
v1 = make_envelope(
    {"name": "Ada Lovelace", "action": "login"},
    version=1,
)

v2 = migrate_envelope(v1)
assert v2["version"] == 2
assert v2["payload"]["first_name"] == "Ada"
assert v2["payload"]["last_name"] == "Lovelace"

print(json.dumps(v2, indent=2))

{
  "schema": "user-event",
  "version": 2,
  "payload": {
    "action": "login",
    "first_name": "Ada",
    "last_name": "Lovelace"
  }
}


# Problem 10 — Atomic JSON file writes

A process crash during `json.dump()` can leave a partially written file.

Write JSON atomically:

1. Create a temporary file in the destination directory.
2. Write UTF-8 JSON.
3. Flush and `fsync`.
4. Replace the destination with `os.replace()`.

This is a common reliability pattern for configuration/state files.

### Solution

In [25]:
def atomic_write_json(path: Path, data: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    fd, temp_name = tempfile.mkstemp(
        prefix=path.name + ".",
        suffix=".tmp",
        dir=path.parent,
        text=True,
    )

    temp_path = Path(temp_name)

    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="\n") as f:
            json.dump(
                data,
                f,
                cls=AdvancedJSONEncoder,
                ensure_ascii=False,
                sort_keys=True,
                indent=2,
                allow_nan=False,
            )
            f.write("\n")
            f.flush()
            os.fsync(f.fileno())

        os.replace(temp_path, path)
    except Exception:
        try:
            temp_path.unlink(missing_ok=True)
        finally:
            raise

In [26]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "state.json"

    state = {
        "saved_at": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
        "balance": Money(Decimal("42.50"), "EUR"),
    }

    atomic_write_json(path, state)

    raw = path.read_text(encoding="utf-8")
    print(raw)

    restored = json.loads(raw, object_hook=decode_json_object)
    assert restored == state

{
  "balance": {
    "__type__": "dataclass",
    "class": "Money",
    "fields": {
      "amount": {
        "__type__": "decimal",
        "value": "42.50"
      },
      "currency": "EUR"
    }
  },
  "saved_at": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  }
}



# Problem 11 — JSON Lines for streaming logs

A single giant JSON array is inconvenient for append-only logs. JSON Lines (`.jsonl`) stores one JSON object per line.

Create:

- `write_jsonl(records, file_obj)`
- `read_jsonl(file_obj)`

Requirements:

- one compact JSON value per line,
- custom types supported,
- malformed line reports its line number.

### Solution

In [27]:
def write_jsonl(records, file_obj) -> None:
    for record in records:
        line = json.dumps(
            record,
            default=encode_json_value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )
        file_obj.write(line + "\n")


def read_jsonl(file_obj):
    for line_number, line in enumerate(file_obj, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            yield json.loads(line, object_hook=decode_json_object)
        except (json.JSONDecodeError, ValueError, TypeError) as exc:
            raise ValueError(
                f"Invalid JSONL record on line {line_number}: {exc}"
            ) from exc

In [28]:
records = [
    {
        "event": "created",
        "at": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
        "id": uuid.UUID("cccccccc-cccc-cccc-cccc-cccccccccccc"),
    },
    {
        "event": "charged",
        "amount": Money(Decimal("12.34"), "EUR"),
        "at": datetime(2026, 8, 7, 12, 1, tzinfo=timezone.utc),
    },
]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "events.jsonl"

    with path.open("w", encoding="utf-8") as f:
        write_jsonl(records, f)

    print(path.read_text(encoding="utf-8"))

    with path.open("r", encoding="utf-8") as f:
        restored = list(read_jsonl(f))

    assert restored == records

{"at":{"__type__":"datetime","value":"2026-08-07T12:00:00Z"},"event":"created","id":{"__type__":"uuid","value":"cccccccc-cccc-cccc-cccc-cccccccccccc"}}
{"amount":{"__type__":"dataclass","class":"Money","fields":{"amount":{"__type__":"decimal","value":"12.34"},"currency":"EUR"}},"at":{"__type__":"datetime","value":"2026-08-07T12:01:00Z"},"event":"charged"}



# Problem 12 — Security: reject dangerous reconstruction patterns

Consider this **bad idea**:

```python
# NEVER DO THIS
type_name = obj["__type__"]
cls = eval(type_name)
return cls(**obj["fields"])
```

Explain the risk, then design a safe alternative.

### Goal

Only explicitly registered dataclasses and Enums may be reconstructed.
Unknown class names must fail.

### Solution

In [29]:
def safe_registered_type_names():
    return {
        "dataclasses": sorted(DATACLASS_REGISTRY),
        "enums": sorted(ENUM_REGISTRY),
    }


print(safe_registered_type_names())

{'dataclasses': ['Money', 'User'], 'enums': ['Status']}


In [30]:
malicious_or_unknown = json.dumps({
    TYPE_KEY: "dataclass",
    "class": "os.system",
    "fields": {"command": "echo should-not-run"},
})

try:
    json.loads(malicious_or_unknown, object_hook=decode_json_object)
except ValueError as exc:
    print("Safely rejected:", exc)
else:
    raise AssertionError("Unknown class name should have been rejected.")

Safely rejected: Dataclass is not whitelisted: 'os.system'


### Security takeaway

JSON parsing should produce **data**, not arbitrary code execution.

Use fixed registries, schema validation, bounded sizes, explicit constructors, no `eval`, no dynamic imports from untrusted type names, and no arbitrary object creation from user-supplied class names.

# Problem 13 — Diagnose collision risk in type-tagged dictionaries

Our format reserves the key `__type__`. But what if normal user data legitimately contains:

```python
{"__type__": "decimal", "value": "999"}
```

`object_hook` would interpret it as a tagged `Decimal`.

Design a safer envelope format that reduces collisions by requiring a dedicated wrapper shape.

### Solution

In [31]:
MARKER_KEY = "__python_json__"


def wrap_tag(tag: str, **payload):
    return {
        MARKER_KEY: {
            "type": tag,
            "payload": payload,
        }
    }


def is_tag_wrapper(obj: dict) -> bool:
    return (
        set(obj) == {MARKER_KEY}
        and isinstance(obj[MARKER_KEY], dict)
        and set(obj[MARKER_KEY]) == {"type", "payload"}
        and isinstance(obj[MARKER_KEY]["type"], str)
        and isinstance(obj[MARKER_KEY]["payload"], dict)
    )

In [32]:
ordinary_user_data = {
    "__type__": "decimal",
    "value": "999",
}

assert not is_tag_wrapper(ordinary_user_data)

wrapped = wrap_tag("decimal", value="999")
assert is_tag_wrapper(wrapped)

print("Ordinary:", ordinary_user_data)
print("Wrapped: ", wrapped)

Ordinary: {'__type__': 'decimal', 'value': '999'}
Wrapped:  {'__python_json__': {'type': 'decimal', 'payload': {'value': '999'}}}


> In high-assurance systems, an even stronger design is to validate the entire JSON document against an explicit schema instead of relying only on ad-hoc markers.

# Problem 14 — Performance trade-off: pretty vs compact JSON

Measure the difference between pretty-printed and compact JSON for a moderately large payload.

Compare:

- serialized size,
- serialization time.

Do not assume one format is always better: pretty JSON is useful for people; compact JSON is useful on the wire.

### Solution

In [33]:
benchmark_payload = [
    {
        "id": i,
        "name": f"user-{i}",
        "price": Decimal(f"{i % 100}.{i % 10}{(i + 3) % 10}"),
        "active": i % 2 == 0,
        "tags": {"json", "python", f"group-{i % 7}"},
    }
    for i in range(1000)
]


def dump_pretty():
    return json.dumps(
        benchmark_payload,
        default=encode_json_value,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )


def dump_compact():
    return json.dumps(
        benchmark_payload,
        default=encode_json_value,
        separators=(",", ":"),
        sort_keys=True,
        ensure_ascii=False,
    )


pretty = dump_pretty()
compact = dump_compact()

print(f"Pretty size : {len(pretty):,} chars")
print(f"Compact size: {len(compact):,} chars")
print(f"Size ratio  : {len(pretty) / len(compact):.2f}x")

Pretty size : 264,182 chars
Compact size: 152,181 chars
Size ratio  : 1.74x


In [34]:
pretty_time = timeit.timeit(dump_pretty, number=10)
compact_time = timeit.timeit(dump_compact, number=10)

print(f"Pretty 10x : {pretty_time:.4f}s")
print(f"Compact 10x: {compact_time:.4f}s")

Pretty 10x : 0.1379s
Compact 10x: 0.1332s


# Problem 15 — Capstone: nested event model with round-trip invariants

Build a realistic event payload containing:

- a dataclass user,
- a dataclass money amount,
- UUIDs,
- timestamp,
- exact decimal values,
- sets,
- bytes,
- fractions,
- nested dictionaries/lists.

Then test important invariants after a JSON round trip.

### Solution

In [35]:
@dataclass
class AuditEvent:
    event_id: uuid.UUID
    actor: User
    occurred_at: datetime
    amount: Money
    metadata: dict


register_dataclass_type(AuditEvent)

__main__.AuditEvent

In [36]:
event = AuditEvent(
    event_id=uuid.UUID("dddddddd-dddd-dddd-dddd-dddddddddddd"),
    actor=User(
        user_id=uuid.UUID("eeeeeeee-eeee-eeee-eeee-eeeeeeeeeeee"),
        name="Grace",
        created_at=datetime(2026, 8, 1, 9, 0, tzinfo=timezone.utc),
        status=Status.ACTIVE,
        tags={"admin", "finance"},
    ),
    occurred_at=datetime(2026, 8, 7, 14, 45, tzinfo=timezone.utc),
    amount=Money(Decimal("1000000.00000001"), "USD"),
    metadata={
        "source": "api",
        "checksum": b"\x10\x20\x30\x40",
        "ratio": Fraction(355, 113),
        "complex_hint": 1.5 - 2.25j,
        "paths": [Path("/var/log/app"), Path("/srv/data")],
        "flags": frozenset({"reviewed", "exported"}),
    },
)

event_json = json.dumps(
    event,
    cls=AdvancedJSONEncoder,
    ensure_ascii=False,
    sort_keys=True,
    indent=2,
    allow_nan=False,
)

print(event_json)

{
  "__type__": "dataclass",
  "class": "AuditEvent",
  "fields": {
    "actor": {
      "__type__": "dataclass",
      "class": "User",
      "fields": {
        "created_at": {
          "__type__": "datetime",
          "value": "2026-08-01T09:00:00Z"
        },
        "name": "Grace",
        "status": {
          "__type__": "enum",
          "enum": "Status",
          "value": "active"
        },
        "tags": {
          "__type__": "set",
          "items": [
            "admin",
            "finance"
          ]
        },
        "user_id": {
          "__type__": "uuid",
          "value": "eeeeeeee-eeee-eeee-eeee-eeeeeeeeeeee"
        }
      }
    },
    "amount": {
      "__type__": "dataclass",
      "class": "Money",
      "fields": {
        "amount": {
          "__type__": "decimal",
          "value": "1000000.00000001"
        },
        "currency": "USD"
      }
    },
    "event_id": {
      "__type__": "uuid",
      "value": "dddddddd-dddd-dddd-dddd-dddddddd

In [37]:
restored_event = json.loads(event_json, object_hook=decode_json_object)

assert restored_event == event
assert isinstance(restored_event, AuditEvent)
assert isinstance(restored_event.actor, User)
assert isinstance(restored_event.actor.user_id, uuid.UUID)
assert isinstance(restored_event.occurred_at, datetime)
assert restored_event.occurred_at.tzinfo is not None
assert isinstance(restored_event.amount.amount, Decimal)
assert restored_event.amount.amount == Decimal("1000000.00000001")
assert restored_event.metadata["ratio"] == Fraction(355, 113)
assert restored_event.metadata["checksum"] == b"\x10\x20\x30\x40"
assert restored_event.metadata["complex_hint"] == 1.5 - 2.25j
assert restored_event.metadata["flags"] == frozenset({"reviewed", "exported"})

print("Capstone round trip passed.")

Capstone round trip passed.


# Problem 16 — Write reusable `dumps` / `loads` façade functions

Consumers should not need to remember every `json.dumps` / `json.loads` option.

Create two small public API functions:

- `dumps(obj, *, pretty=False)`
- `loads(text)`

Goals:

- centralized policy,
- strict numbers,
- Unicode-friendly output,
- deterministic keys,
- custom encoding/decoding.

### Solution

In [38]:
def dumps(obj: Any, *, pretty: bool = False) -> str:
    kwargs = {
        "cls": AdvancedJSONEncoder,
        "ensure_ascii": False,
        "sort_keys": True,
        "allow_nan": False,
    }

    if pretty:
        kwargs["indent"] = 2
    else:
        kwargs["separators"] = (",", ":")

    return json.dumps(obj, **kwargs)


def loads(text: str) -> Any:
    return json.loads(
        text,
        object_hook=decode_json_object,
        parse_float=Decimal,
    )

In [39]:
text = dumps(event, pretty=False)
copy = loads(text)

assert copy == event
print(text[:300] + "...")
print("Reusable façade functions passed.")

{"__type__":"dataclass","class":"AuditEvent","fields":{"actor":{"__type__":"dataclass","class":"User","fields":{"created_at":{"__type__":"datetime","value":"2026-08-01T09:00:00Z"},"name":"Grace","status":{"__type__":"enum","enum":"Status","value":"active"},"tags":{"__type__":"set","items":["admin","...
Reusable façade functions passed.


# Problem 17 — Failure-mode tests

A production serializer should be tested on bad inputs, not just good inputs.

Write tests for:

- unsupported object type,
- unknown tag,
- malformed Base64,
- unknown Enum,
- unknown dataclass,
- naive datetime,
- NaN rejection.

### Solution

In [40]:
class Unsupported:
    pass


failure_cases_passed = 0

try:
    dumps({"x": Unsupported()})
except TypeError:
    failure_cases_passed += 1

try:
    json.loads('{"__type__":"mystery","value":1}', object_hook=decode_json_object)
except ValueError:
    failure_cases_passed += 1

bad_b64 = json.dumps({
    "__type__": "bytes",
    "encoding": "base64",
    "value": "***not-base64***",
})
try:
    json.loads(bad_b64, object_hook=decode_json_object)
except ValueError:
    failure_cases_passed += 1

bad_enum = json.dumps({
    "__type__": "enum",
    "enum": "DangerousEnum",
    "value": "x",
})
try:
    json.loads(bad_enum, object_hook=decode_json_object)
except ValueError:
    failure_cases_passed += 1

bad_dc = json.dumps({
    "__type__": "dataclass",
    "class": "SecretClass",
    "fields": {},
})
try:
    json.loads(bad_dc, object_hook=decode_json_object)
except ValueError:
    failure_cases_passed += 1

try:
    dumps({"dt": datetime(2026, 1, 1)})
except ValueError:
    failure_cases_passed += 1

try:
    dumps({"x": math.nan})
except ValueError:
    failure_cases_passed += 1

assert failure_cases_passed == 7
print("All 7 failure-mode checks passed.")

All 7 failure-mode checks passed.


# Problem 18 — Property-style round-trip testing without external libraries

Generate many representative values and assert:

```python
loads(dumps(value)) == value
```

This is not a replacement for Hypothesis/property-based testing, but it teaches the same invariant-driven mindset using only the standard library.

### Solution

In [41]:
round_trip_values = [
    Decimal("0"),
    Decimal("-999999999999999999.0001"),
    Fraction(0, 1),
    Fraction(-22, 7),
    5 + 0j,
    -3.25 + 8.5j,
    b"",
    bytes(range(16)),
    uuid.UUID(int=0),
    Path("."),
    set(),
    {"a", "b", "c"},
    frozenset({1, 2, 3}),
    datetime(2026, 1, 1, tzinfo=timezone.utc),
    datetime(2026, 12, 31, 23, 59, 59, 999999, tzinfo=timezone.utc),
    date(2026, 8, 7),
    time(23, 59, 59, 123456),
    Status.CLOSED,
    Money(Decimal("12.3400"), "BGN"),
]

for index, value in enumerate(round_trip_values, start=1):
    restored = loads(dumps(value))
    assert restored == value, (index, value, restored)
    assert type(restored) is type(value), (
        index,
        type(value).__name__,
        type(restored).__name__,
    )

print(f"{len(round_trip_values)} round-trip invariant checks passed.")

19 round-trip invariant checks passed.


# Additional Challenge A — Why `vars()` as a fallback is risky

Consider a `Session` object containing an access token and internal cache.

### Exercise

Write a safe explicit serializer for `Session` that exposes only `"user"`.

### Solution

In [42]:
class Session:
    def __init__(self, user, access_token):
        self.user = user
        self.access_token = access_token
        self.cache = {"internal": True}


@encode_json_value.register(Session)
def _(obj):
    return {
        TYPE_KEY: "session_public",
        "user": obj.user,
    }


session_json = dumps(Session("ada", "SECRET-TOKEN"))
print(session_json)

assert "SECRET-TOKEN" not in session_json
assert "cache" not in session_json

{"__type__":"session_public","user":"ada"}


# Additional Challenge B — Preserve tuple-vs-list distinction

Python's JSON encoder automatically converts tuples to JSON arrays *before* `default=` is called, so a normal `default` hook cannot distinguish a tuple from a list.

### Question

How can you preserve tuple type?

### Answer

Preprocess the object graph before calling `json.dumps`, or use a custom representation at the domain-model level.

In [43]:
def preprocess_tuples(obj):
    if isinstance(obj, tuple):
        return {
            TYPE_KEY: "tuple",
            "items": [preprocess_tuples(item) for item in obj],
        }

    if isinstance(obj, list):
        return [preprocess_tuples(item) for item in obj]

    if isinstance(obj, dict):
        return {
            key: preprocess_tuples(value)
            for key, value in obj.items()
        }

    if isinstance(obj, set):
        return {
            TYPE_KEY: "set",
            "items": [preprocess_tuples(item) for item in obj],
        }

    return obj

In [44]:
_decode_before_tuple = decode_json_object

def decode_json_object_with_tuple(obj):
    if obj.get(TYPE_KEY) == "tuple":
        return tuple(obj["items"])
    return _decode_before_tuple(obj)


value = {
    "list": [1, 2, 3],
    "tuple": (1, 2, 3),
    "nested": [(4, 5), [6, 7]],
}

preprocessed = preprocess_tuples(value)
text = json.dumps(preprocessed, default=encode_json_value, sort_keys=True)
restored = json.loads(text, object_hook=decode_json_object_with_tuple)

assert isinstance(restored["list"], list)
assert isinstance(restored["tuple"], tuple)
assert isinstance(restored["nested"][0], tuple)
assert isinstance(restored["nested"][1], list)

print(text)

{"list": [1, 2, 3], "nested": [{"__type__": "tuple", "items": [4, 5]}, [6, 7]], "tuple": {"__type__": "tuple", "items": [1, 2, 3]}}


# Additional Challenge C — Detect circular references

JSON cannot represent arbitrary object graphs with cycles without introducing an identity/reference scheme.

Demonstrate the failure and discuss design choices.

In [45]:
cyclic = []
cyclic.append(cyclic)

try:
    json.dumps(cyclic)
except ValueError as exc:
    print("Expected circular-reference failure:", exc)

Expected circular-reference failure: Circular reference detected


### Design options for graphs

If your domain requires cycles/shared references, decide whether to:

- reject them,
- flatten the graph into tables keyed by IDs,
- encode `{"$ref": "object-id"}` references,
- use a graph-oriented format instead of plain JSON.

Do not casually disable `check_circular`; that can turn a clean error into unbounded recursion.

# Final Review

You now have multiple serialization strategies:

- `default=` callable
- `@singledispatch`
- `JSONEncoder` subclass
- tagged round-trip decoding with `object_hook`
- explicit dataclass/Enum registries
- deterministic output
- atomic persistence
- JSON Lines streaming
- schema versioning
- failure-mode and invariant testing

## Recommended production baseline

1. Keep the public JSON schema simple.
2. Prefer plain JSON-native structures at service boundaries.
3. Use explicit tagged values only when you truly need Python-type round trips.
4. Use aware UTC datetimes.
5. Preserve exact decimal values intentionally.
6. Make serializers strict.
7. Whitelist reconstructed classes.
8. Version durable formats.
9. Add negative tests.
10. Treat serialization as part of your API contract.